In [ ]:
""" Run Instructions:
1. Run step by step
2. Modify variables throughtout depending on desired timeframe and data
3. Script works with "Fitted" and "Ne from Power" files. Does not work with "Resolved Velocity"
"""
import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import madrigalWeb.madrigalWeb
import os
import h5py
import numpy as np

In [ ]:
#CHANGE ME
user_fullname = "Benjamin Marcotte"
user_email = "bmarc@bu.edu"
user_affiliation = "ISR Summer School 2026"

maddat = madrigalWeb.madrigalWeb.MadrigalData('https://data.amisr.com/madrigal/')
#maddat = madrigalWeb.madrigalWeb.MadrigalData('https://cedar.openmadrigal.org/')

In [ ]:
#instrument codes for AMISR
instcodes={'PFISR':61,
          'RISR-N':91,
          'RISR-C':92}

In [ ]:
# Find an experiment that happened in the following time interval:
st=datetime.datetime(2026, 7, 20, 0,0)
et=datetime.datetime(2026, 7, 22, 23,59)

expList = maddat.getExperiments(instcodes['PFISR'],
                st.year, st.month, st.day, st.hour, st.minute, st.second,
                et.year, et.month, et.day, et.hour, et.minute, et.second)
for exp in expList:
    print(exp)

In [ ]:
# Input Desired Experiment ID from list above into variable below:
selected_experiment_id = 30003611
fileList = maddat.getExperimentFiles(selected_experiment_id)
for file in fileList:
    print(os.path.basename(file.name),'\tkindat:',file.kindat,'desc:',file.kindatdesc)

In [ ]:
# Input desired kindat value from list above into variable below:
# Works with "Fitted" and "Ne From Power" files. Does not work with "Resolved Velocity"
selected_file_kindat = 1000303
acfile=None
for file in fileList:
    if file.kindat == selected_file_kindat:
        acfile=file
        break
        
filename    = acfile.name
outfilename = os.path.basename(acfile.name)
result = maddat.downloadFile(filename,outfilename, user_fullname, user_email, user_affiliation, 'hdf5')
print(f"Done downloading {outfilename}")

In [ ]:
# Explore the HDF5 madrigal metadata
with h5py.File(outfilename,'r') as f:
    for key1,val1 in f.items():
        print(key1,val1)
        for key2,val2 in val1.items():
            print(" ",key2,val2)        

In [ ]:
# Explore the HDF5 madrigal data from the first beam in /Data/Array Layout
with h5py.File(outfilename,'r') as f:
    for key1,val1 in f["/Data/Array Layout"].items():
        print(key1,val1)
        for key2,val2 in val1.items():
            print(" ",key2,val2) 
            try:
                for key3,val3 in val2.items():
                    print("   ",key3,val3)      
            except:
                pass
        break

In [ ]:
if "nenotr" in outfilename:
    with h5py.File(outfilename,'r') as f:
        PFISR_data = []
        for dat in f['Data/Array Layout'].values():
            outdct={}
            outdct['bid'] = dat['1D Parameters/beamid'][0]
            outdct['azm'] = dat['1D Parameters/azm'][0]
            outdct['elm'] = dat['1D Parameters/elm'][0]
            outdct['ne'] = 10**(dat['2D Parameters/popl'][:])
            outdct['dne'] = 10**(dat['2D Parameters/dpopl'][:])
            
            outdct['range'] = dat['range'][:]
            outdct['altitude'] = outdct['range']*np.sin(np.radians(outdct['elm']))
            tstmp = dat['timestamps'][:]
            outdct['time'] = [datetime.datetime.fromtimestamp(t, tz = datetime.timezone.utc) for t in tstmp]
            PFISR_data.append(outdct) 
elif "fit" in outfilename:
    with h5py.File(outfilename,'r') as f:
        PFISR_data = []
        for dat in f['Data/Array Layout'].values():
            outdct={}
            outdct['bid'] = dat['1D Parameters/beamid'][0]
            outdct['azm'] = dat['1D Parameters/azm'][0]
            outdct['elm'] = dat['1D Parameters/elm'][0]
            outdct['ne'] = dat['2D Parameters/ne'][:]     # different from old SRI madrigal 2
            outdct['dne'] = dat['2D Parameters/dne'][:]   # different from old SRI madrigal 2
            outdct['te'] = dat['2D Parameters/te'][:]
            outdct['dte'] = dat['2D Parameters/dte'][:]
            outdct['ti'] = dat['2D Parameters/ti'][:]
            outdct['dti'] = dat['2D Parameters/dti'][:]
            outdct['vo'] = dat['2D Parameters/vo'][:]
            outdct['dvo'] = dat['2D Parameters/dvo'][:]
            
            outdct['range'] = dat['range'][:]
            outdct['altitude'] = outdct['range']*np.sin(np.radians(outdct['elm']))
            tstmp = dat['timestamps'][:]
            outdct['time'] = [datetime.datetime.fromtimestamp(t, tz = datetime.timezone.utc) for t in tstmp]
            PFISR_data.append(outdct)
else:
    print("Cannot currently handle this file type.")

In [ ]:
for i,d in enumerate(PFISR_data):
    print(d['bid'],d['azm'],d['elm'])

In [ ]:
# Input index of desired beam (starting at 0) from list above into variable below
selected_beam_id = 0
beam_data = PFISR_data[selected_beam_id]

In [ ]:
fig,ax=plt.subplots(figsize=(8,6), dpi=130)
clrs = ax.pcolormesh(mdates.date2num(beam_data['time']),
                     beam_data['altitude'],
                     beam_data['ne'],
                     vmin=0,vmax=2e11,shading='nearest')

locator = mdates.AutoDateLocator(minticks=3, maxticks=7)
formatter = mdates.ConciseDateFormatter(locator)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)

ax.set_xlabel('UT')
ax.set_ylabel('Altitude (km)')
ax.set_title('Electron Density vs Range') # Use metadata to make more descriptive

cb=fig.colorbar(clrs)
cb.set_label('Ne (m$^{-3}$)')

In [ ]:
# Enter cutoff times below
start_time_year = 2026
start_time_month = 7
start_time_day = 21
start_time_hour = 10
start_time_minute = 0

end_time_year = 2026
end_time_month = 7
end_time_day = 21
end_time_hour = 11
end_time_minute = 0

start_time = datetime.datetime(
    start_time_year, start_time_month, start_time_day, start_time_hour, start_time_minute, 0,
    tzinfo=datetime.timezone.utc
)

# Find the datetime with the smallest absolute time difference (now including month/day)
start_time_index = min(
    range(len(beam_data['time'])),
    key=lambda i: abs((beam_data['time'][i] - start_time).total_seconds())
)

end_time = datetime.datetime(
    end_time_year, end_time_month, end_time_day, end_time_hour, end_time_minute, 0,
    tzinfo=datetime.timezone.utc
)

end_time_index = min(
    range(len(beam_data['time'])),
    key=lambda i: abs((beam_data['time'][i] - end_time).total_seconds())
)

fig,ax=plt.subplots(figsize=(8,6), dpi=130)
clrs = ax.pcolormesh(mdates.date2num(beam_data['time'][start_time_index:end_time_index]),
                     beam_data['altitude'],
                     beam_data['ne'][:, start_time_index:end_time_index],
                     vmin=0,vmax=2e11,shading='nearest')

locator = mdates.AutoDateLocator(minticks=3, maxticks=7)
formatter = mdates.ConciseDateFormatter(locator)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(formatter)

ax.set_xlabel('UT')
ax.set_ylabel('Altitude (km)')
ax.set_title('Electron Density vs Range') # Use metadata to make more descriptive

cb=fig.colorbar(clrs)
cb.set_label('Ne (m$^{-3}$)')